# 60 - Build Gold Job Cost

Builds a Power BI-ready Job Cost star using the conformed customer, location, employee, and date dimensions.

In [1]:
# Attach the schema-enabled Lakehouse containing the Bronze shortcuts and
# make it the default Lakehouse before running this notebook.
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"
GOLD_SCHEMA = "gold"
METADATA_SCHEMA = "metadata"
COMPANY_KEY = "NAC"  # Change when processing another GP company database.

def q(identifier: str) -> str:
    return f"`{identifier.replace('`', '``')}`"

def sql_string(value: str) -> str:
    return "'" + value.replace("'", "''") + "'"

def table_exists(schema_name: str, table_name: str) -> bool:
    try:
        return spark.catalog.tableExists(f"{schema_name}.{table_name}")
    except Exception:
        return False

def require_tables(table_names: list[str], schema_name: str = BRONZE_SCHEMA) -> None:
    missing = [name for name in table_names if not table_exists(schema_name, name)]
    if missing:
        raise RuntimeError(
            f"Missing tables in {schema_name}: {', '.join(missing)}. "
            "Confirm the OneLake shortcuts and exact table names."
        )

company = sql_string(COMPANY_KEY)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(SILVER_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(GOLD_SCHEMA)}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {q(METADATA_SCHEMA)}")
print(f"Company: {COMPANY_KEY}; Bronze schema: {BRONZE_SCHEMA}")

StatementMeta(, 6820b955-4eb8-43f6-b657-e25bbe25731b, 3, Finished, Available, Finished, False)

Company: NAC; Bronze schema: bronze


In [2]:
require_tables(["job", "job_date", "job_people", "job_financial_snapshot",
                "job_cost", "job_billing"], SILVER_SCHEMA)
require_tables(["dim_customer", "dim_location", "dim_employee", "dim_date"], GOLD_SCHEMA)

StatementMeta(, 6820b955-4eb8-43f6-b657-e25bbe25731b, 4, Finished, Available, Finished, False)

## Job dimension

In [3]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.dim_job
USING DELTA
AS
SELECT CAST(0 AS BIGINT) AS job_key, {company} AS company_key,
       'UNKNOWN' AS job_number, 'Unknown job' AS job_name,
       CAST(NULL AS STRING) AS job_description, CAST(NULL AS STRING) AS job_status,
       CAST(NULL AS STRING) AS division_code, CAST(NULL AS STRING) AS division_name,
       CAST(NULL AS BIGINT) AS customer_key, CAST(NULL AS BIGINT) AS location_key,
       CAST(NULL AS INT) AS scheduled_start_date_key,
       CAST(NULL AS INT) AS scheduled_completion_date_key,
       CAST(NULL AS INT) AS actual_completion_date_key
UNION ALL
SELECT
    XXHASH64(j.company_key, j.job_number) AS job_key,
    j.company_key, j.job_number, j.job_name, j.job_description, j.job_status,
    j.division_code, j.division_name,
    COALESCE(c.customer_key, 0), COALESCE(l.location_key, 0),
    TRY_CAST(DATE_FORMAT(d.scheduled_start_date, 'yyyyMMdd') AS INT),
    TRY_CAST(DATE_FORMAT(d.scheduled_completion_date, 'yyyyMMdd') AS INT),
    TRY_CAST(DATE_FORMAT(d.actual_completion_date, 'yyyyMMdd') AS INT)
FROM {q(SILVER_SCHEMA)}.job j
LEFT JOIN {q(SILVER_SCHEMA)}.job_date d
  ON j.company_key = d.company_key AND j.job_number = d.job_number
LEFT JOIN {q(GOLD_SCHEMA)}.dim_customer c
  ON j.company_key = c.company_key AND j.customer_code = c.customer_code
LEFT JOIN {q(GOLD_SCHEMA)}.dim_location l
  ON j.company_key = l.company_key AND j.customer_code = l.customer_code
 AND j.location_code = l.location_code
""")

StatementMeta(, 6820b955-4eb8-43f6-b657-e25bbe25731b, 5, Finished, Available, Finished, False)

DataFrame[]

## Job financial snapshot and transaction facts

In [4]:
spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_job_financial_snapshot
USING DELTA
AS
SELECT
    XXHASH64(f.company_key, f.job_number, CAST(f.snapshot_date AS STRING)) AS job_snapshot_key,
    COALESCE(j.job_key, 0) AS job_key,
    f.*
FROM {q(SILVER_SCHEMA)}.job_financial_snapshot f
LEFT JOIN {q(GOLD_SCHEMA)}.dim_job j
  ON f.company_key = j.company_key AND f.job_number = j.job_number
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_job_cost
USING DELTA
AS
SELECT
    XXHASH64(c.company_key, c.job_number, CAST(c.journal_number AS STRING),
              CAST(c.cost_code AS STRING), CAST(c.document_date AS STRING)) AS job_cost_key,
    COALESCE(j.job_key, 0) AS job_key,
    c.company_key, c.job_number, c.vendor_code, c.journal_number,
    c.cost_code, c.cost_label, c.document_code,
    TRY_CAST(DATE_FORMAT(c.document_date, 'yyyyMMdd') AS INT) AS document_date_key,
    c.document_amount,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.job_cost c
LEFT JOIN {q(GOLD_SCHEMA)}.dim_job j
  ON c.company_key = j.company_key AND c.job_number = j.job_number
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {q(GOLD_SCHEMA)}.fact_job_billing
USING DELTA
AS
SELECT
    XXHASH64(b.company_key, b.job_number, b.document_number,
              CAST(b.document_date AS STRING)) AS job_billing_key,
    COALESCE(j.job_key, 0) AS job_key,
    COALESCE(c.customer_key, 0) AS customer_key,
    b.company_key, b.job_number, b.document_number,
    TRY_CAST(DATE_FORMAT(b.document_date, 'yyyyMMdd') AS INT) AS document_date_key,
    b.document_amount,
    CURRENT_TIMESTAMP() AS _processed_at
FROM {q(SILVER_SCHEMA)}.job_billing b
LEFT JOIN {q(GOLD_SCHEMA)}.dim_job j
  ON b.company_key = j.company_key AND b.job_number = j.job_number
LEFT JOIN {q(GOLD_SCHEMA)}.dim_customer c
  ON b.company_key = c.company_key AND b.customer_code = c.customer_code
""")

StatementMeta(, 6820b955-4eb8-43f6-b657-e25bbe25731b, 6, Finished, Available, Finished, False)

DataFrame[]

## Review

In [5]:
for name in ["dim_job", "fact_job_financial_snapshot", "fact_job_cost", "fact_job_billing"]:
    print(f"{GOLD_SCHEMA}.{name}: {spark.table(f'{GOLD_SCHEMA}.{name}').count():,} rows")

StatementMeta(, 6820b955-4eb8-43f6-b657-e25bbe25731b, 7, Finished, Available, Finished, False)

gold.dim_job: 4,481 rows
gold.fact_job_financial_snapshot: 4,480 rows
gold.fact_job_cost: 972 rows
gold.fact_job_billing: 686 rows
